# AutoInt+ 실험 노트북

이 노트북 하나로 실험 조합 선택, variant 데이터 생성, AutoInt+ 학습, 평가, 결과표 반영까지 수행합니다.

## 실행 방식

1. `RUN_ALL_EXPERIMENTS` 또는 `EXPERIMENT_IDS`를 설정합니다.
2. `SKIP_COMPLETED = True`이면 결과 CSV에서 이미 `완료` 상태인 실험은 자동으로 건너뜁니다.
3. 실험 하나가 끝날 때마다 즉시 CSV/Markdown 결과표가 저장됩니다.
4. 중간에 끊긴 뒤 다시 실행해도 완료된 실험은 재실행하지 않습니다.

In [7]:
import sys
from importlib.util import find_spec

PROJECT_ROOT = "/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(sys.path[:5])
print(find_spec("autoint"))

from autoint.tf_device import configure_tensorflow
configure_tensorflow()


['/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel', '/Users/jkyu/miniforge3/envs/ds8_rcmm/lib/python311.zip', '/Users/jkyu/miniforge3/envs/ds8_rcmm/lib/python3.11', '/Users/jkyu/miniforge3/envs/ds8_rcmm/lib/python3.11/lib-dynload', '']
ModuleSpec(name='autoint', loader=<_frozen_importlib_external.SourceFileLoader object at 0x11143d610>, origin='/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/autoint/__init__.py', submodule_search_locations=['/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/autoint'])
[tf] GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [8]:
from pathlib import Path
import importlib.util
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "experiments").exists() and (candidate / "autoint").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


ROOT_DIR = find_project_root()
EXPERIMENTS_DIR = ROOT_DIR / "experiments"
RESULTS_DIR = EXPERIMENTS_DIR / "results"
ARTIFACTS_DIR = EXPERIMENTS_DIR / "artifacts"
RAW_DATA_DIR = ROOT_DIR / "autoint" / "data" / "ml-1m"

RESULTS_CSV = RESULTS_DIR / "autoint_plus_results_template.csv"
RESULTS_MD = RESULTS_DIR / "autoint_plus_results.md"

USERS_PREPRO_CSV = RAW_DATA_DIR / "users_prepro.csv"
RATINGS_PREPRO_CSV = RAW_DATA_DIR / "ratings_prepro.csv"
MOVIES_PREPRO_CSV = RAW_DATA_DIR / "movies_prepro.csv"

BASE_FEATURE_COLUMNS = [
    "user_id",
    "movie_id",
    "movie_decade",
    "movie_year",
    "rating_year",
    "rating_month",
    "rating_decade",
    "genre1",
    "genre2",
    "genre3",
    "gender",
    "age",
    "occupation",
    "zip",
]

EXPERIMENTS = [
    {
        "experiment_id": "E0",
        "preference_rule": "p0_rating_ge_4",
        "recency_rule": "r0_all_history",
        "genre_rewatch_feature": "g0_off",
        "preference_desc": "평점 4점 이상을 선호로 간주",
        "recency_desc": "사용자 전체 이력 사용",
        "genre_rewatch_desc": "장르 재시청 피처 사용 안 함",
    },
    {
        "experiment_id": "E1",
        "preference_rule": "p1_clean_preference",
        "recency_rule": "r0_all_history",
        "genre_rewatch_feature": "g0_off",
        "preference_desc": "4~5점은 선호, 1~2점은 비선호, 3점은 제외",
        "recency_desc": "사용자 전체 이력 사용",
        "genre_rewatch_desc": "장르 재시청 피처 사용 안 함",
    },
    {
        "experiment_id": "E2",
        "preference_rule": "p0_rating_ge_4",
        "recency_rule": "r1_last_6_months",
        "genre_rewatch_feature": "g0_off",
        "preference_desc": "평점 4점 이상을 선호로 간주",
        "recency_desc": "사용자별 최근 6개월 이력만 사용",
        "genre_rewatch_desc": "장르 재시청 피처 사용 안 함",
    },
    {
        "experiment_id": "E3",
        "preference_rule": "p0_rating_ge_4",
        "recency_rule": "r0_all_history",
        "genre_rewatch_feature": "g1_on",
        "preference_desc": "평점 4점 이상을 선호로 간주",
        "recency_desc": "사용자 전체 이력 사용",
        "genre_rewatch_desc": "장르 재시청 여부를 이진 피처로 추가",
    },
    {
        "experiment_id": "E4",
        "preference_rule": "p1_clean_preference",
        "recency_rule": "r1_last_6_months",
        "genre_rewatch_feature": "g0_off",
        "preference_desc": "4~5점은 선호, 1~2점은 비선호, 3점은 제외",
        "recency_desc": "사용자별 최근 6개월 이력만 사용",
        "genre_rewatch_desc": "장르 재시청 피처 사용 안 함",
    },
    {
        "experiment_id": "E5",
        "preference_rule": "p1_clean_preference",
        "recency_rule": "r0_all_history",
        "genre_rewatch_feature": "g1_on",
        "preference_desc": "4~5점은 선호, 1~2점은 비선호, 3점은 제외",
        "recency_desc": "사용자 전체 이력 사용",
        "genre_rewatch_desc": "장르 재시청 여부를 이진 피처로 추가",
    },
    {
        "experiment_id": "E6",
        "preference_rule": "p0_rating_ge_4",
        "recency_rule": "r1_last_6_months",
        "genre_rewatch_feature": "g1_on",
        "preference_desc": "평점 4점 이상을 선호로 간주",
        "recency_desc": "사용자별 최근 6개월 이력만 사용",
        "genre_rewatch_desc": "장르 재시청 여부를 이진 피처로 추가",
    },
    {
        "experiment_id": "E7",
        "preference_rule": "p1_clean_preference",
        "recency_rule": "r1_last_6_months",
        "genre_rewatch_feature": "g1_on",
        "preference_desc": "4~5점은 선호, 1~2점은 비선호, 3점은 제외",
        "recency_desc": "사용자별 최근 6개월 이력만 사용",
        "genre_rewatch_desc": "장르 재시청 여부를 이진 피처로 추가",
    },
]

RESULT_COLUMNS = [
    "experiment_id",
    "preference_rule",
    "recency_rule",
    "genre_rewatch_feature",
    "preference_desc",
    "recency_desc",
    "genre_rewatch_desc",
    "dataset_name",
    "train_rows",
    "val_rows",
    "test_rows",
    "train_positive_ratio",
    "val_positive_ratio",
    "test_positive_ratio",
    "val_bce",
    "ndcg_at_10",
    "hitrate_at_10",
    "status",
    "notes",
]

TABLE_COLUMNS = [
    ("experiment_id", "실험"),
    ("preference_desc", "선호 기준"),
    ("recency_desc", "최근성 조건"),
    ("genre_rewatch_desc", "장르 재시청"),
    ("dataset_name", "데이터셋"),
    ("train_rows", "학습 행 수"),
    ("val_rows", "검증 행 수"),
    ("test_rows", "테스트 행 수"),
    ("train_positive_ratio", "학습 Positive 비율"),
    ("val_bce", "검증 BCE"),
    ("ndcg_at_10", "NDCG@10"),
    ("hitrate_at_10", "HitRate@10"),
    ("status", "상태"),
    ("notes", "비고"),
]

print(ROOT_DIR)

/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel


In [9]:
# 실행 설정
RUN_ALL_EXPERIMENTS = True
EXPERIMENT_IDS = ["E0"]
SKIP_COMPLETED = True
EPOCHS = 5
BATCH_SIZE = 2048
LEARNING_RATE = 0.0001
EMBED_DIM = 16
TOP_K = 10
SEED = 42
SAVE_MODEL_ARTIFACTS = True

if RUN_ALL_EXPERIMENTS:
    EXPERIMENT_IDS = [item["experiment_id"] for item in EXPERIMENTS]

EXPERIMENT_IDS = list(dict.fromkeys(EXPERIMENT_IDS))
print("실행 대상:", EXPERIMENT_IDS)
print("완료 실험 스킵:", SKIP_COMPLETED)

실행 대상: ['E0', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7']
완료 실험 스킵: True


In [10]:
def get_experiment_config(experiment_id: str) -> dict:
    config_map = {item["experiment_id"]: item for item in EXPERIMENTS}
    if experiment_id not in config_map:
        raise ValueError(f"알 수 없는 실험 ID입니다: {experiment_id}")
    return config_map[experiment_id].copy()


def ensure_preprocessed_inputs_exist() -> None:
    required = [USERS_PREPRO_CSV, RATINGS_PREPRO_CSV, MOVIES_PREPRO_CSV]
    missing = [path.as_posix() for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("전처리 입력 파일이 없습니다: " + ", ".join(missing))


def load_preprocessed_frames() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ensure_preprocessed_inputs_exist()
    users_df = pd.read_csv(USERS_PREPRO_CSV)
    ratings_df = pd.read_csv(RATINGS_PREPRO_CSV)
    movies_df = pd.read_csv(MOVIES_PREPRO_CSV)
    return users_df, ratings_df, movies_df


def apply_preference_rule(ratings_df: pd.DataFrame, rule: str) -> pd.DataFrame:
    working = ratings_df.copy()
    if rule == "p0_rating_ge_4":
        working["label"] = working["rating"].ge(4).astype(int)
        return working
    if rule == "p1_clean_preference":
        working = working[working["rating"] != 3].copy()
        working["label"] = working["rating"].ge(4).astype(int)
        return working
    raise ValueError(f"지원하지 않는 선호 기준입니다: {rule}")


def apply_recency_rule(ratings_df: pd.DataFrame, rule: str) -> pd.DataFrame:
    working = ratings_df.copy()
    working["_timestamp_dt"] = pd.to_datetime(working["timestamp"])
    if rule == "r0_all_history":
        return working
    if rule == "r1_last_6_months":
        cutoff = working.groupby("user_id")["_timestamp_dt"].transform(
            lambda series: series.max() - pd.DateOffset(months=6)
        )
        return working[working["_timestamp_dt"] >= cutoff].copy()
    raise ValueError(f"지원하지 않는 최근성 조건입니다: {rule}")


def normalize_genre_values(values) -> set[str]:
    genres = set()
    for value in values:
        text = str(value).strip()
        if not text or text.lower() in {"nan", "none", "no"}:
            continue
        genres.add(text)
    return genres


def add_genre_rewatch_feature(merged_df: pd.DataFrame) -> pd.DataFrame:
    working = merged_df.sort_values(["user_id", "_timestamp_dt", "movie_id"]).copy()
    working["genre_rewatch"] = 0
    for _, group in working.groupby("user_id", sort=False):
        seen_positive_genres = set()
        indices = []
        values = []
        for row in group.itertuples():
            genres = normalize_genre_values([row.genre1, row.genre2, row.genre3])
            values.append(1 if genres & seen_positive_genres else 0)
            if int(row.label) == 1:
                seen_positive_genres.update(genres)
            indices.append(row.Index)
        working.loc[indices, "genre_rewatch"] = values
    return working


def build_variant_dataset(config: dict) -> tuple[pd.DataFrame, dict]:
    users_df, ratings_df, movies_df = load_preprocessed_frames()
    ratings_variant = apply_recency_rule(ratings_df, config["recency_rule"])
    ratings_variant = apply_preference_rule(ratings_variant, config["preference_rule"])
    ratings_variant = ratings_variant[
        [
            "user_id",
            "movie_id",
            "rating",
            "timestamp",
            "_timestamp_dt",
            "rating_year",
            "rating_month",
            "rating_decade",
            "label",
        ]
    ].copy()
    movies_variant = movies_df[["movie_id", "movie_decade", "movie_year", "genre1", "genre2", "genre3"]].copy()
    users_variant = users_df[["user_id", "gender", "age", "occupation", "zip"]].copy()
    merged_df = pd.merge(ratings_variant, movies_variant, on="movie_id", how="left")
    merged_df = pd.merge(merged_df, users_variant, on="user_id", how="left")
    merged_df.fillna("no", inplace=True)
    feature_columns = list(BASE_FEATURE_COLUMNS)
    if config["genre_rewatch_feature"] == "g1_on":
        merged_df = add_genre_rewatch_feature(merged_df)
        feature_columns.append("genre_rewatch")
    dataset_df = merged_df[feature_columns + ["label"]].copy()
    metadata = {
        "dataset_name": f"{config['experiment_id']}_autoint_plus_dataset.csv",
        "rows": int(len(dataset_df)),
        "positive_ratio": round(float(dataset_df['label'].mean()), 6),
        "feature_columns": feature_columns,
    }
    return dataset_df, metadata


def load_tensorflow_modules():
    try:
        import tensorflow as tf
        from autoint.tf_device import configure_tensorflow
        from tensorflow.keras.losses import BinaryCrossentropy
        from tensorflow.keras.optimizers import Adam
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "TensorFlow가 설치되어 있지 않아 실험을 실행할 수 없습니다. 먼저 `pip install -r requirements.txt` 또는 TensorFlow 설치를 진행하세요."
        ) from exc
    configure_tensorflow()
    return tf, Adam, BinaryCrossentropy


def load_autoint_plus_model_class():
    module_path = ROOT_DIR / "autoint" / "autointmlp.py"
    spec = importlib.util.spec_from_file_location("autointmlp_experiment_module", module_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"모델 파일을 불러올 수 없습니다: {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.AutoIntMLPModel


def prepare_encoded_dataset(dataset_df: pd.DataFrame):
    working = dataset_df.copy()
    feature_columns = list(working.columns[:-1])
    label_column = working.columns[-1]
    label_encoders = {column: LabelEncoder() for column in feature_columns}
    for column, encoder in label_encoders.items():
        working[column] = encoder.fit_transform(working[column].astype(str))
    working[label_column] = working[label_column].astype(np.float32)
    field_dims = np.max(working[feature_columns].astype(np.int64).values, axis=0) + 1
    return working, feature_columns, label_encoders, field_dims


def split_dataset(dataset_df: pd.DataFrame, seed: int):
    train_val_df, test_df = train_test_split(dataset_df, test_size=0.2, random_state=seed)
    train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=seed)
    return train_df, val_df, test_df


def get_dcg(ranklist: list[int], y_true: list[int]) -> float:
    dcg = 0.0
    for index, item in enumerate(ranklist):
        if item in y_true:
            dcg += 1.0 / np.log(index + 2)
    return dcg


def get_idcg(y_true: list[int]) -> float:
    idcg = 0.0
    for index, _ in enumerate(y_true):
        idcg += 1.0 / np.log(index + 2)
    return idcg


def get_ndcg(ranklist: list[int], y_true: list[int]) -> float:
    if not y_true:
        return 0.0
    dcg = get_dcg(ranklist, y_true)
    idcg = get_idcg(y_true)
    if idcg == 0:
        return 0.0
    return round(float(dcg / idcg), 5)


def get_hit_rate(ranklist: list[int], y_true: list[int]) -> float:
    if not y_true:
        return 0.0
    hits = sum(1 for item in y_true if item in ranklist)
    return round(float(hits / len(y_true)), 5)


def rank_predictions(model, test_df: pd.DataFrame, feature_columns: list[str], top_k: int):
    user_pred_info = {}
    model_user_pred_info = {}
    batch_size = 2048
    total_rows = len(test_df)
    for start in range(0, total_rows, batch_size):
        batch = test_df.iloc[start:start + batch_size]
        features = batch[feature_columns].values
        predictions = model.predict(features, verbose=False)
        for feature, score in zip(features, predictions):
            user_id = int(feature[0])
            movie_id = int(feature[1])
            model_user_pred_info.setdefault(user_id, []).append((movie_id, float(score)))
    for user_id, pairs in model_user_pred_info.items():
        ranklist = sorted(pairs, key=lambda item: item[1], reverse=True)[:top_k]
        user_pred_info[str(user_id)] = list(dict.fromkeys([movie_id for movie_id, _ in ranklist]))
    test_positive = test_df[test_df["label"] == 1].groupby("user_id")["movie_id"].apply(list)
    ndcg_scores = {}
    hitrate_scores = {}
    for user_id, movie_ids in test_positive.items():
        predicted = user_pred_info.get(str(user_id), [])[:top_k]
        truth = list(set(np.array(movie_ids).astype(int)))
        ndcg_scores[user_id] = get_ndcg(predicted, truth)
        hitrate_scores[user_id] = get_hit_rate(predicted, truth)
    ndcg = round(float(np.mean(list(ndcg_scores.values()))), 5) if ndcg_scores else 0.0
    hitrate = round(float(np.mean(list(hitrate_scores.values()))), 5) if hitrate_scores else 0.0
    return ndcg, hitrate


def initialize_results_df() -> pd.DataFrame:
    rows = []
    for experiment in EXPERIMENTS:
        row = {column: "" for column in RESULT_COLUMNS}
        row.update(experiment)
        row["status"] = "대기"
        rows.append(row)
    summary = {column: "" for column in RESULT_COLUMNS}
    summary.update(
        {
            "experiment_id": "summary",
            "preference_rule": "best_to_be_filled",
            "recency_rule": "best_to_be_filled",
            "genre_rewatch_feature": "best_to_be_filled",
            "preference_desc": "아직 완료된 실험이 없습니다",
            "recency_desc": "아직 완료된 실험이 없습니다",
            "genre_rewatch_desc": "아직 완료된 실험이 없습니다",
            "status": "대기",
            "notes": "실험 결과가 입력되면 자동으로 최고 성능 조합을 표시합니다.",
        }
    )
    rows.append(summary)
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)


def load_results_df() -> pd.DataFrame:
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV, dtype=str).fillna("")
    return initialize_results_df()


def save_results_df(results_df: pd.DataFrame) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_CSV, index=False)


def update_summary_row(results_df: pd.DataFrame) -> pd.DataFrame:
    working = results_df.copy()
    completed = working[
        (working["experiment_id"] != "summary")
        & (working["status"].astype(str).str.strip() == "완료")
        & (working["ndcg_at_10"].astype(str).str.strip() != "")
        & (working["hitrate_at_10"].astype(str).str.strip() != "")
    ].copy()
    if completed.empty:
        working.loc[working["experiment_id"] == "summary", ["preference_desc", "recency_desc", "genre_rewatch_desc", "status", "notes"]] = [
            "아직 완료된 실험이 없습니다",
            "아직 완료된 실험이 없습니다",
            "아직 완료된 실험이 없습니다",
            "대기",
            "실험 결과가 입력되면 자동으로 최고 성능 조합을 표시합니다.",
        ]
        return working
    completed["_ndcg"] = pd.to_numeric(completed["ndcg_at_10"])
    completed["_hitrate"] = pd.to_numeric(completed["hitrate_at_10"])
    completed["_val_bce"] = pd.to_numeric(completed["val_bce"], errors="coerce").fillna(999999.0)
    best = completed.sort_values(["_ndcg", "_hitrate", "_val_bce"], ascending=[False, False, True]).iloc[0]
    mask = working["experiment_id"] == "summary"
    for column in RESULT_COLUMNS:
        if column == "experiment_id":
            continue
        if column == "status":
            working.loc[mask, column] = "요약 완료"
        elif column == "notes":
            working.loc[mask, column] = f"현재 최고 실험은 {best['experiment_id']}입니다. NDCG@10={best['ndcg_at_10']}, HitRate@10={best['hitrate_at_10']}."
        else:
            working.loc[mask, column] = best.get(column, "")
    return working


def render_results_markdown(results_df: pd.DataFrame) -> str:
    headers = [label for _, label in TABLE_COLUMNS]
    lines = [
        "# AutoInt+ 실험 결과 비교표",
        "",
        f"원본 CSV: `{RESULTS_CSV.as_posix()}`",
        "",
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in results_df.iterrows():
        rendered = []
        for key, _ in TABLE_COLUMNS:
            value = str(row.get(key, "")).strip()
            rendered.append(value if value else "-")
        lines.append("| " + " | ".join(rendered) + " |")
    return "\n".join(lines) + "\n"


def save_results_outputs(results_df: pd.DataFrame) -> None:
    save_results_df(results_df)
    RESULTS_MD.write_text(render_results_markdown(results_df), encoding="utf-8")


def is_completed_experiment(results_df: pd.DataFrame, experiment_id: str) -> bool:
    row = results_df[results_df["experiment_id"] == experiment_id]
    if row.empty:
        return False
    row = row.iloc[0]
    return (
        str(row.get("status", "")).strip() == "완료"
        and str(row.get("ndcg_at_10", "")).strip() != ""
        and str(row.get("hitrate_at_10", "")).strip() != ""
    )


def mark_experiment_running(results_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    working = results_df.copy()
    mask = working["experiment_id"] == config["experiment_id"]
    for key, value in config.items():
        working.loc[mask, key] = value
    working.loc[mask, "status"] = "실행 중"
    working.loc[mask, "notes"] = "실험 실행 중입니다. 중단되면 다시 실행 시 이어서 남은 실험만 진행합니다."
    return working


def mark_experiment_interrupted(results_df: pd.DataFrame, experiment_id: str, message: str) -> pd.DataFrame:
    working = results_df.copy()
    mask = working["experiment_id"] == experiment_id
    working.loc[mask, "status"] = "중단됨"
    working.loc[mask, "notes"] = message
    return working


def run_single_experiment(config: dict, tf, Adam, BinaryCrossentropy, AutoIntMLPModel) -> dict:
    dataset_df, dataset_meta = build_variant_dataset(config)
    encoded_df, feature_columns, label_encoders, field_dims = prepare_encoded_dataset(dataset_df)
    train_df, val_df, test_df = split_dataset(encoded_df, SEED)
    tf.keras.utils.set_random_seed(SEED)

    model = AutoIntMLPModel(
        field_dims=field_dims,
        embedding_size=EMBED_DIM,
        att_layer_num=3,
        att_head_num=2,
        att_res=True,
        dnn_hidden_units=(32, 32),
        dnn_activation="relu",
        l2_reg_dnn=0,
        l2_reg_embedding=1e-5,
        dnn_use_bn=False,
        dnn_dropout=0.4,
        init_std=0.0001,
    )
    model(tf.constant([[0] * len(field_dims)], dtype=tf.int64))

    optimizer = Adam(learning_rate=LEARNING_RATE)
    loss_fn = BinaryCrossentropy(from_logits=False)
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=["binary_crossentropy"])

    history = model.fit(
        train_df[feature_columns],
        train_df["label"],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(val_df[feature_columns], val_df["label"]),
        verbose=1,
    )

    val_metrics = model.evaluate(val_df[feature_columns], val_df["label"], verbose=0, return_dict=True)
    ndcg_at_10, hitrate_at_10 = rank_predictions(model, test_df, feature_columns, TOP_K)

    metrics = {
        "dataset_name": dataset_meta["dataset_name"],
        "train_rows": str(len(train_df)),
        "val_rows": str(len(val_df)),
        "test_rows": str(len(test_df)),
        "train_positive_ratio": f"{float(train_df['label'].mean()):.6f}",
        "val_positive_ratio": f"{float(val_df['label'].mean()):.6f}",
        "test_positive_ratio": f"{float(test_df['label'].mean()):.6f}",
        "val_bce": f"{float(val_metrics.get('binary_crossentropy', val_metrics.get('loss', 0.0))):.5f}",
        "ndcg_at_10": f"{ndcg_at_10:.5f}",
        "hitrate_at_10": f"{hitrate_at_10:.5f}",
        "status": "완료",
        "notes": "노트북 실행으로 갱신됨",
    }

    artifact_dir = ARTIFACTS_DIR / config["experiment_id"]
    artifact_dir.mkdir(parents=True, exist_ok=True)
    if SAVE_MODEL_ARTIFACTS:
        model.save_weights(artifact_dir / "autoint_plus.weights.h5")
        np.save(artifact_dir / "field_dims.npy", field_dims)
        joblib.dump(label_encoders, artifact_dir / "label_encoders.pkl")
        (artifact_dir / "history.json").write_text(json.dumps(history.history, indent=2, ensure_ascii=False), encoding="utf-8")
        (artifact_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

    return {
        "config": config,
        "dataset_meta": dataset_meta,
        "metrics": metrics,
        "artifact_dir": artifact_dir.as_posix(),
    }


In [11]:
selected_configs = [get_experiment_config(experiment_id) for experiment_id in EXPERIMENT_IDS]
display(pd.DataFrame(EXPERIMENTS))
display(pd.DataFrame(selected_configs))

,experiment_id,preference_rule,recency_rule,genre_rewatch_feature,preference_desc,recency_desc,genre_rewatch_desc
0,E0,p0_rating_ge_4,r0_all_history,g0_off,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 피처 사용 안 함
1,E1,p1_clean_preference,r0_all_history,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 피처 사용 안 함
2,E2,p0_rating_ge_4,r1_last_6_months,g0_off,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함
3,E3,p0_rating_ge_4,r0_all_history,g1_on,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가
4,E4,p1_clean_preference,r1_last_6_months,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함
5,E5,p1_clean_preference,r0_all_history,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가
6,E6,p0_rating_ge_4,r1_last_6_months,g1_on,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가
7,E7,p1_clean_preference,r1_last_6_months,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가


,experiment_id,preference_rule,recency_rule,genre_rewatch_feature,preference_desc,recency_desc,genre_rewatch_desc
0,E0,p0_rating_ge_4,r0_all_history,g0_off,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 피처 사용 안 함
1,E1,p1_clean_preference,r0_all_history,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 피처 사용 안 함
2,E2,p0_rating_ge_4,r1_last_6_months,g0_off,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함
3,E3,p0_rating_ge_4,r0_all_history,g1_on,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가
4,E4,p1_clean_preference,r1_last_6_months,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함
5,E5,p1_clean_preference,r0_all_history,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가
6,E6,p0_rating_ge_4,r1_last_6_months,g1_on,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가
7,E7,p1_clean_preference,r1_last_6_months,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가


In [12]:
tf, Adam, BinaryCrossentropy = load_tensorflow_modules()
AutoIntMLPModel = load_autoint_plus_model_class()

results_df = load_results_df()
run_outputs = []
skipped_ids = []

for config in selected_configs:
    experiment_id = config["experiment_id"]

    if SKIP_COMPLETED and is_completed_experiment(results_df, experiment_id):
        skipped_ids.append(experiment_id)
        print(f"[{experiment_id}] 이미 완료된 실험이라 스킵합니다.")
        continue

    print(f"\n=== {experiment_id} 시작 ===")
    results_df = mark_experiment_running(results_df, config)
    results_df = update_summary_row(results_df)
    save_results_outputs(results_df)

    try:
        output = run_single_experiment(config, tf, Adam, BinaryCrossentropy, AutoIntMLPModel)
    except BaseException as exc:
        message = f"실험이 중단되었거나 실패했습니다: {type(exc).__name__}: {exc}"
        results_df = mark_experiment_interrupted(results_df, experiment_id, message)
        results_df = update_summary_row(results_df)
        save_results_outputs(results_df)
        raise

    run_outputs.append(output)

    for key, value in config.items():
        results_df.loc[results_df["experiment_id"] == experiment_id, key] = value
    for key, value in output["metrics"].items():
        results_df.loc[results_df["experiment_id"] == experiment_id, key] = value
    results_df.loc[results_df["experiment_id"] == experiment_id, "notes"] = (
        f"가중치와 메타데이터는 {output['artifact_dir']}에 저장했습니다."
    )
    results_df = update_summary_row(results_df)
    save_results_outputs(results_df)

    print(
        f"[{experiment_id}] val_bce={output['metrics']['val_bce']}, "
        f"ndcg@10={output['metrics']['ndcg_at_10']}, hitrate@10={output['metrics']['hitrate_at_10']}"
    )

if run_outputs:
    summary_df = pd.DataFrame([
        {
            **output["config"],
            **output["metrics"],
            "artifact_dir": output["artifact_dir"],
        }
        for output in run_outputs
    ])
    display(summary_df)
else:
    print("이번 실행에서 새로 완료된 실험은 없습니다.")

print("스킵된 실험:", skipped_ids)

[tf] GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

=== E0 시작 ===


2026-04-10 14:50:06.809943: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-04-10 14:50:06.809972: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-04-10 14:50:06.809976: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 13.50 GB
I0000 00:00:1775800206.810008 13798137 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1775800206.810044 13798137 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/5


2026-04-10 14:50:10.027060: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


352/352 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - binary_crossentropy: 0.6754 - loss: 0.6754 - val_binary_crossentropy: 0.6467 - val_loss: 0.6467
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - binary_crossentropy: 0.6169 - loss: 0.6169 - val_binary_crossentropy: 0.5942 - val_loss: 0.5942
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - binary_crossentropy: 0.5635 - loss: 0.5635 - val_binary_crossentropy: 0.5525 - val_loss: 0.5525
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - binary_crossentropy: 0.5424 - loss: 0.5424 - val_binary_crossentropy: 0.5470 - val_loss: 0.5470
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - binary_crossentropy: 0.5371 - loss: 0.5371 - val_binary_crossentropy: 0.5447 - val_loss: 0.5447


...

[E6] val_bce=0.55082, ndcg@10=0.71496, hitrate@10=0.70405

=== E7 시작 ===
Epoch 1/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step - binary_crossentropy: 0.6152 - loss: 0.6152 - val_binary_crossentropy: 0.5268 - val_loss: 0.5268
Epoch 2/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - binary_crossentropy: 0.5179 - loss: 0.5179 - val_binary_crossentropy: 0.5124 - val_loss: 0.5124
Epoch 3/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - binary_crossentropy: 0.4861 - loss: 0.4861 - val_binary_crossentropy: 0.4567 - val_loss: 0.4567
Epoch 4/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - binary_crossentropy: 0.4254 - loss: 0.4254 - val_binary_crossentropy: 0.4040 - val_loss: 0.4040
Epoch 5/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - binary_crossentropy: 0.3916 - loss: 0.3916 - val_binary_crossentropy: 0.3861 - val_loss: 0.3861


/var/folders/q3/hz0fg47j4_g8nqffrmt9hlc00000gn/T/ipykernel_33946/3437901787.py:197: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  model_user_pred_info.setdefault(user_id, []).append((movie_id, float(score)))
/var/folders/q3/hz0fg47j4_g8nqffrmt9hlc00000gn/T/ipykernel_33946/3437901787.py:197: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  model_user_pred_info.setdefault(user_id, []).append((movie_id, float(score)))
/var/folders/q3/hz0fg47j4_g8nqffrmt9hlc00000gn/T/ipykernel_33946/3437901787.py:197: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element 

[E7] val_bce=0.38610, ndcg@10=0.79305, hitrate@10=0.75297


,experiment_id,preference_rule,recency_rule,genre_rewatch_feature,preference_desc,recency_desc,genre_rewatch_desc,dataset_name,train_rows,val_rows,test_rows,train_positive_ratio,val_positive_ratio,test_positive_ratio,val_bce,ndcg_at_10,hitrate_at_10,status,notes,artifact_dir
0,E0,p0_rating_ge_4,r0_all_history,g0_off,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 피처 사용 안 함,E0_autoint_plus_dataset.csv,720150,80017,200042,0.574964,0.575615,0.575689,0.54474,0.66221,0.63022,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
1,E1,p1_clean_preference,r0_all_history,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 피처 사용 안 함,E1_autoint_plus_dataset.csv,532088,59121,147803,0.778187,0.778776,0.779247,0.35974,0.73929,0.68310,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
2,E2,p0_rating_ge_4,r1_last_6_months,g0_off,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함,E2_autoint_plus_dataset.csv,528599,58734,146834,0.576125,0.575646,0.573048,0.54856,0.71606,0.70441,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
3,E3,p0_rating_ge_4,r0_all_history,g1_on,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가,E3_autoint_plus_dataset.csv,720150,80017,200042,0.575318,0.573391,0.575304,0.54347,0.66403,0.63102,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
4,E4,p1_clean_preference,r1_last_6_months,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함,E4_autoint_plus_dataset.csv,390695,43411,108527,0.778042,0.780885,0.779677,0.37760,0.79291,0.75308,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
5,E5,p1_clean_preference,r0_all_history,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가,E5_autoint_plus_dataset.csv,532088,59121,147803,0.778027,0.778928,0.779761,0.36271,0.73782,0.68185,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
6,E6,p0_rating_ge_4,r1_last_6_months,g1_on,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가,E6_autoint_plus_dataset.csv,528599,58734,146834,0.575675,0.576089,0.574492,0.55082,0.71496,0.70405,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...
7,E7,p1_clean_preference,r1_last_6_months,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가,E7_autoint_plus_dataset.csv,390695,43411,108527,0.778592,0.776854,0.779308,0.38610,0.79305,0.75297,완료,노트북 실행으로 갱신됨,/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_r...


스킵된 실험: []


In [13]:
preview_ids = EXPERIMENT_IDS + ["summary"]
display(pd.read_csv(RESULTS_CSV).query("experiment_id in @preview_ids"))
print(RESULTS_MD.read_text(encoding='utf-8')[:4000])

,experiment_id,preference_rule,recency_rule,genre_rewatch_feature,preference_desc,recency_desc,genre_rewatch_desc,dataset_name,train_rows,val_rows,test_rows,train_positive_ratio,val_positive_ratio,test_positive_ratio,val_bce,ndcg_at_10,hitrate_at_10,status,notes
0,E0,p0_rating_ge_4,r0_all_history,g0_off,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 피처 사용 안 함,E0_autoint_plus_dataset.csv,720150,80017,200042,0.574964,0.575615,0.575689,0.54474,0.66221,0.63022,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
1,E1,p1_clean_preference,r0_all_history,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 피처 사용 안 함,E1_autoint_plus_dataset.csv,532088,59121,147803,0.778187,0.778776,0.779247,0.35974,0.73929,0.68310,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
2,E2,p0_rating_ge_4,r1_last_6_months,g0_off,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함,E2_autoint_plus_dataset.csv,528599,58734,146834,0.576125,0.575646,0.573048,0.54856,0.71606,0.70441,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
3,E3,p0_rating_ge_4,r0_all_history,g1_on,평점 4점 이상을 선호로 간주,사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가,E3_autoint_plus_dataset.csv,720150,80017,200042,0.575318,0.573391,0.575304,0.54347,0.66403,0.63102,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
4,E4,p1_clean_preference,r1_last_6_months,g0_off,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 피처 사용 안 함,E4_autoint_plus_dataset.csv,390695,43411,108527,0.778042,0.780885,0.779677,0.37760,0.79291,0.75308,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
5,E5,p1_clean_preference,r0_all_history,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자 전체 이력 사용,장르 재시청 여부를 이진 피처로 추가,E5_autoint_plus_dataset.csv,532088,59121,147803,0.778027,0.778928,0.779761,0.36271,0.73782,0.68185,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
6,E6,p0_rating_ge_4,r1_last_6_months,g1_on,평점 4점 이상을 선호로 간주,사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가,E6_autoint_plus_dataset.csv,528599,58734,146834,0.575675,0.576089,0.574492,0.55082,0.71496,0.70405,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
7,E7,p1_clean_preference,r1_last_6_months,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가,E7_autoint_plus_dataset.csv,390695,43411,108527,0.778592,0.776854,0.779308,0.38610,0.79305,0.75297,완료,가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/Ma...
8,summary,p1_clean_preference,r1_last_6_months,g1_on,"4~5점은 선호, 1~2점은 비선호, 3점은 제외",사용자별 최근 6개월 이력만 사용,장르 재시청 여부를 이진 피처로 추가,E7_autoint_plus_dataset.csv,390695,43411,108527,0.778592,0.776854,0.779308,0.38610,0.79305,0.75297,요약 완료,"현재 최고 실험은 E7입니다. NDCG@10=0.79305, HitRate@10=0..."


# AutoInt+ 실험 결과 비교표

원본 CSV: `/Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/experiments/results/autoint_plus_results_template.csv`

| 실험 | 선호 기준 | 최근성 조건 | 장르 재시청 | 데이터셋 | 학습 행 수 | 검증 행 수 | 테스트 행 수 | 학습 Positive 비율 | 검증 BCE | NDCG@10 | HitRate@10 | 상태 | 비고 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| E0 | 평점 4점 이상을 선호로 간주 | 사용자 전체 이력 사용 | 장르 재시청 피처 사용 안 함 | E0_autoint_plus_dataset.csv | 720150 | 80017 | 200042 | 0.574964 | 0.54474 | 0.66221 | 0.63022 | 완료 | 가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/experiments/artifacts/E0에 저장했습니다. |
| E1 | 4~5점은 선호, 1~2점은 비선호, 3점은 제외 | 사용자 전체 이력 사용 | 장르 재시청 피처 사용 안 함 | E1_autoint_plus_dataset.csv | 532088 | 59121 | 147803 | 0.778187 | 0.35974 | 0.73929 | 0.68310 | 완료 | 가중치와 메타데이터는 /Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/experiments/artifacts/E1에 저장했습니다. |
| E2 | 평점 4점 이상을 선호로 간주 | 사용자별 최근 6개월 이력만 사용 | 장르 재시청 피처 사용 안 함 | E2_autoint_plus_data

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "experiments").exists() and (candidate / "autoint").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

ROOT_DIR = find_project_root()
RESULTS_CSV = ROOT_DIR / "experiments" / "results" / "autoint_plus_results_template.csv"

viz_df = pd.read_csv(RESULTS_CSV).fillna("")
viz_df = viz_df[(viz_df["experiment_id"] != "summary") & (viz_df["status"] == "완료")].copy()

if viz_df.empty:
    print("아직 완료된 실험 결과가 없습니다. 실험 실행 후 다시 시각화하세요.")
else:
    numeric_cols = ["val_bce", "ndcg_at_10", "hitrate_at_10"]
    for col in numeric_cols:
        viz_df[col] = pd.to_numeric(viz_df[col], errors="coerce")

    preference_map = {
        "p0_rating_ge_4": "4점 이상 선호",
        "p1_clean_preference": "1-2 비선호, 3 제외",
    }
    recency_map = {
        "r0_all_history": "전체 이력",
        "r1_last_6_months": "최근 6개월",
    }
    genre_map = {
        "g0_off": "사용 안 함",
        "g1_on": "사용",
    }

    viz_df["preference"] = viz_df["preference_rule"].map(preference_map).fillna(viz_df["preference_rule"])
    viz_df["recency"] = viz_df["recency_rule"].map(recency_map).fillna(viz_df["recency_rule"])
    viz_df["genre_rewatch"] = viz_df["genre_rewatch_feature"].map(genre_map).fillna(viz_df["genre_rewatch_feature"])
    viz_df = viz_df.sort_values(["ndcg_at_10", "hitrate_at_10", "val_bce"], ascending=[False, False, True])

    display(
        viz_df[[
            "experiment_id",
            "preference",
            "recency",
            "genre_rewatch",
            "val_bce",
            "ndcg_at_10",
            "hitrate_at_10",
        ]]
    )

    metric_df = viz_df.melt(
        id_vars=["experiment_id", "preference", "recency", "genre_rewatch"],
        value_vars=["val_bce", "ndcg_at_10", "hitrate_at_10"],
        var_name="metric",
        value_name="value",
    )
    metric_name_map = {
        "val_bce": "Val BCE",
        "ndcg_at_10": "NDCG@10",
        "hitrate_at_10": "HitRate@10",
    }
    metric_df["metric"] = metric_df["metric"].map(metric_name_map).fillna(metric_df["metric"])

    fig_bar = px.bar(
        metric_df,
        x="experiment_id",
        y="value",
        color="metric",
        barmode="group",
        text="value",
        hover_data=["preference", "recency", "genre_rewatch"],
        title="실험별 핵심 지표 비교",
    )
    fig_bar.update_traces(texttemplate="%{text:.4f}", textposition="outside")
    fig_bar.update_layout(xaxis_title="Experiment", yaxis_title="Value")

    figure_dir = ROOT_DIR / "experiments" / "results" / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    figure_html_path = figure_dir / "autoint_plus_experiment_bar.html"
    figure_png_path = figure_dir / "autoint_plus_experiment_bar.png"

    fig_bar.write_html(figure_html_path)
    try:
        fig_bar.write_image(figure_png_path, scale=2)
        print(f"PNG 저장 완료: {figure_png_path}")
    except Exception as exc:
        print(f"PNG 저장 실패: {exc}")
        print("`pip install kaleido` 후 다시 실행하면 PNG 저장이 가능합니다.")
    print(f"HTML 저장 완료: {figure_html_path}")
    fig_bar.show()



,experiment_id,preference,recency,genre_rewatch,val_bce,ndcg_at_10,hitrate_at_10
7,E7,"1-2 비선호, 3 제외",최근 6개월,사용,0.38610,0.79305,0.75297
4,E4,"1-2 비선호, 3 제외",최근 6개월,사용 안 함,0.37760,0.79291,0.75308
1,E1,"1-2 비선호, 3 제외",전체 이력,사용 안 함,0.35974,0.73929,0.68310
5,E5,"1-2 비선호, 3 제외",전체 이력,사용,0.36271,0.73782,0.68185
2,E2,4점 이상 선호,최근 6개월,사용 안 함,0.54856,0.71606,0.70441
6,E6,4점 이상 선호,최근 6개월,사용,0.55082,0.71496,0.70405
3,E3,4점 이상 선호,전체 이력,사용,0.54347,0.66403,0.63102
0,E0,4점 이상 선호,전체 이력,사용 안 함,0.54474,0.66221,0.63022


PNG 저장 완료: /Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/experiments/results/figures/autoint_plus_experiment_bar.png
HTML 저장 완료: /Users/jkyu/aiffel/AIFFEL_QUEST/MainQuest/10_rcm_node_aiffel/experiments/results/figures/autoint_plus_experiment_bar.html
